In [2]:
import sqlite3
import logging
import os

# Set up logging
logging.basicConfig(filename='database_operations.log', level=logging.INFO, 
                    format='%(asctime)s - %(levelname)s - %(message)s')

def connect_to_db(db_path):
    try:
        if not os.path.exists(db_path):
            logging.info(f"Creating new database: {db_path}")
        conn = sqlite3.connect(db_path)
        logging.info(f"Successfully connected to {db_path}")
        return conn
    except sqlite3.Error as e:
        logging.error(f"Error connecting to {db_path}: {e}")
        raise

# Connect to databases
try:
    sdm_conn = connect_to_db('GreatOutdoors_SDM.db')
    dwh_conn = connect_to_db('GreatOutdoors_DWH.db')
    # Note: Keep connections open for use in subsequent cells
except Exception as e:
    logging.critical(f"Failed to establish database connections: {e}")
    # Handle or re-raise as needed

In [ ]:
dwh_conn.executescript("""
            -- Dimension / lookup tables first (no FK dependencies)

            CREATE TABLE IF NOT EXISTS DATE  (
                DATE_KEY        INT             NOT NULL,
                YEAR            INT             NOT NULL,
                MONTH           INT             NOT NULL,
                DAY             INT             NOT NULL,
                CONSTRAINT PK_DATE PRIMARY KEY (DATE_KEY)
            );

            CREATE TABLE IF NOT EXISTS CUSTOMER  (
                CUSTOMER_CODE               VARCHAR(50)     NOT NULL,
                COUNTRY                     VARCHAR(100),
                CITY                        VARCHAR(100),
                REGION                      VARCHAR(100),
                POSTAL_ZONE                 VARCHAR(20),
                ACTIVE_INDICATOR            CHAR(1),
                COMPANY_NAME                VARCHAR(200),
                SEGMENT_NAME                VARCHAR(100),
                CUSTOMER_TYPE_EN            VARCHAR(100),
                CUSTOMER_HEADQUARTERS_CODE  VARCHAR(50),
                TERRITORY                   VARCHAR(100),
                CUSTOMER_SITE_CODE          VARCHAR(50),
                EFFECTIVE_FROM              DATE,
                EFFECTIVE_TO                DATE,
                ACTIVE                      CHAR(1),
                CONSTRAINT PK_CUSTOMER PRIMARY KEY (CUSTOMER_CODE)
            );

            CREATE TABLE IF NOT EXISTS PRODUCT  (
                PRODUCT_NUMBER          VARCHAR(50)     NOT NULL,
                INTRODUCTION_DATE       DATE,
                AGE                     INT,
                PRODUCTION_COST_GROUP   VARCHAR(50),
                MARGIN_GROUP            VARCHAR(50),
                PRODUCT_NAME            VARCHAR(200),
                PRODUCT_LINE_EN         VARCHAR(100),
                PRODUCT_TYPE_EN         VARCHAR(100),
                EFFECTIVE_FROM              DATE,
                EFFECTIVE_TO                DATE,
                ACTIVE                      CHAR(1),
                CONSTRAINT PK_PRODUCT PRIMARY KEY (PRODUCT_NUMBER)
            );

            CREATE TABLE IF NOT EXISTS CUSTOMER_AGE_GROUP  (
                AGE_GROUP_CODE  VARCHAR(50)     NOT NULL,
                UPPER_AGE       INT,
                LOWER_AGE       INT,
                CONSTRAINT PK_CUSTOMER_AGE_GROUP PRIMARY KEY (AGE_GROUP_CODE)
            );

            -- Tables with FK dependencies

            CREATE TABLE IF NOT EXISTS ORDER_DETAILS  (
                ORDER_DETAIL_CODE   VARCHAR(50)     NOT NULL,
                ORDER_NUMBER        VARCHAR(50),
                DATE_KEY            INT,
                ORDER_METHOD        VARCHAR(50),
                CUSTOMER_CODE       VARCHAR(50),
                PRODUCT_NUMBER      VARCHAR(50),
                QUANTITY            INT,
                UNIT_COST           DECIMAL(18,2),
                UNIT_PRICE          DECIMAL(18,2),
                UNIT_SALE_PRICE     DECIMAL(18,2),
                OMZET               DECIMAL(18,2),
                WINST               DECIMAL(18,2),
                CONTRIBUTION_MARGIN DECIMAL(18,2),
                CONSTRAINT PK_ORDER_DETAILS PRIMARY KEY (ORDER_DETAIL_CODE),
                CONSTRAINT FK_OD_DATE     FOREIGN KEY (DATE_KEY)       REFERENCES DATE (DATE_KEY),
                CONSTRAINT FK_OD_CUSTOMER FOREIGN KEY (CUSTOMER_CODE)  REFERENCES CUSTOMER (CUSTOMER_CODE),
                CONSTRAINT FK_OD_PRODUCT  FOREIGN KEY (PRODUCT_NUMBER) REFERENCES PRODUCT (PRODUCT_NUMBER)
            );

            CREATE TABLE IF NOT EXISTS RETURNED_ITEM  (
                RETURN_CODE         VARCHAR(50)     NOT NULL,
                RETURN_DATE         DATE,
                RETURN_REASON       VARCHAR(200),
                ORDER_DETAIL_CODE   VARCHAR(50),
                RETURN_QUANTITY     INT,
                CONSTRAINT PK_RETURNED_ITEM PRIMARY KEY (RETURN_CODE),
                CONSTRAINT FK_RI_ORDER_DETAIL FOREIGN KEY (ORDER_DETAIL_CODE) REFERENCES ORDER_DETAILS (ORDER_DETAIL_CODE)
            );

            CREATE TABLE IF NOT EXISTS SALES_DEMOGRAPHIC  (
                DEMOGRAPHIC_CODE    VARCHAR(50)     NOT NULL,
                CUSTOMER_CODE       VARCHAR(50),
                AGE_GROUP_CODE      VARCHAR(50),
                SALES_PERCENT       DECIMAL(10,4),
                CONSTRAINT PK_SALES_DEMOGRAPHIC PRIMARY KEY (DEMOGRAPHIC_CODE),
                CONSTRAINT FK_SD_CUSTOMER      FOREIGN KEY (CUSTOMER_CODE)  REFERENCES CUSTOMER (CUSTOMER_CODE),
                CONSTRAINT FK_SD_AGE_GROUP     FOREIGN KEY (AGE_GROUP_CODE) REFERENCES CUSTOMER_AGE_GROUP (AGE_GROUP_CODE)
            );

            CREATE TABLE IF NOT EXISTS INVENTORY  (
                DATE_KEY            INT             NOT NULL,
                PRODUCT_KEY         VARCHAR(50)     NOT NULL,
                INVENTORY_COUNT     INT,
                CONSTRAINT PK_INVENTORY PRIMARY KEY (DATE_KEY, PRODUCT_KEY),
                CONSTRAINT FK_INV_DATE    FOREIGN KEY (DATE_KEY)    REFERENCES DATE (DATE_KEY),
                CONSTRAINT FK_INV_PRODUCT FOREIGN KEY (PRODUCT_KEY) REFERENCES PRODUCT (PRODUCT_NUMBER)
            );

            CREATE TABLE IF NOT EXISTS PRODUCT_FORECAST  (
                PRODUCT_KEY         VARCHAR(50)     NOT NULL,
                DATE_KEY            INT             NOT NULL,
                EXPECTED_VOLUME     DECIMAL(18,2),
                CONSTRAINT PK_PRODUCT_FORECAST PRIMARY KEY (PRODUCT_KEY, DATE_KEY),
                CONSTRAINT FK_PF_PRODUCT FOREIGN KEY (PRODUCT_KEY) REFERENCES PRODUCT (PRODUCT_NUMBER),
                CONSTRAINT FK_PF_DATE    FOREIGN KEY (DATE_KEY)    REFERENCES DATE (DATE_KEY)
            );
            """)